In [ ]:
#!/usr/bin/env python3
"""
Compute **Multi‑Level Order‑Flow Imbalance (OFI)** for a single stock.

The implementation mirrors the style and column‑naming scheme of the
*best‑level* script you provided, but generalises it to the first *M* levels
(m = 0, 1, …, M‑1).  Per‑update flows follow Eq. (2) and per‑interval
normalisation follows Eq. (3) of *“Cross‑Impact of Order‑Flow Imbalance”*.

---------------------------------------------------------------------------
"""

from __future__ import annotations

from pathlib import Path
from typing import List

import numpy as np
import pandas as pd


# ── CONFIGURE INPUT / OUTPUT PATHS ──────────────────────────────────────────
LOB_CSV: Path = Path("first_25000_rows.csv")      # raw LOB updates
OUT_CSV: Path = Path("multi_level_ofi.csv")       # final feature matrix
BUCKET:  str  = "1min"                            # resampling frequency

# Column name helpers (consistent with best‑level script)
COL_TIME = "ts_event"                             # timestamp column


def _px(level: int, side: str) -> str:
    """Return bid/ask price column name at book level *level* (0‑indexed)."""
    return f"{side}_px_{level:02d}"


def _sz(level: int, side: str) -> str:
    """Return bid/ask size column name at book level *level* (0‑indexed)."""
    return f"{side}_sz_{level:02d}"


# ── CORE FUNCTIONS ─────────────────────────────────────────────────────────


def load_lob(path: Path, levels: int) -> pd.DataFrame:
    """
    Read the raw LOB CSV (first *levels* tiers) and return a chronologically
    sorted DataFrame with parsed UTC timestamps.
    """
    cols: List[str] = [COL_TIME]
    for m in range(levels):
        cols += [_px(m, "bid"), _sz(m, "bid"), _px(m, "ask"), _sz(m, "ask")]

    df = pd.read_csv(path, usecols=cols)

    # Parse timestamps (ISO string or epoch‑ms integer)
    if np.issubdtype(df[COL_TIME].dtype, np.number):
        df[COL_TIME] = pd.to_datetime(df[COL_TIME].astype("int64"),
                                      unit="ms", utc=True)
    else:
        df[COL_TIME] = pd.to_datetime(df[COL_TIME], utc=True, errors="coerce")

    df = df.dropna(subset=[COL_TIME]).sort_values(COL_TIME, ignore_index=True)
    return df


def compute_multi_level_ofi(df: pd.DataFrame, levels: int) -> pd.DataFrame:
    """
    Vectorised multi‑level OFI with normalisation by average depth.

    Returns a DataFrame whose DateTimeIndex is the right edge of each *BUCKET*
    interval and whose columns are ``ofi_level_00 … ofi_level_{levels‑1:02d}``.
    """
    resampled_ofi: dict[str, pd.Series] = {}

    # -------- per‑update depth (for normalisation later) -------------------
    depth_per_update = pd.Series(0.0, index=df.index)
    for m in range(levels):
        depth_per_update += df[_sz(m, "bid")] + df[_sz(m, "ask")]

    # -------- compute raw OFI for each level -------------------------------
    for m in range(levels):
        bp, bs = _px(m, "bid"), _sz(m, "bid")
        ap, a_s = _px(m, "ask"), _sz(m, "ask")

        prev = df[[bp, bs, ap, a_s]].shift()

        bid_flow = (
            (df[bp] > prev[bp]) * df[bs] +
            (df[bp] == prev[bp]) * (df[bs] - prev[bs]) +
            (df[bp] < prev[bp]) * (-prev[bs])
        )

        ask_flow = (
            (df[ap] > prev[ap]) * (-df[a_s]) +
            (df[ap] == prev[ap]) * (df[a_s] - prev[a_s]) +
            (df[ap] < prev[ap]) * (prev[a_s])
        )

        raw_ofi = bid_flow - ask_flow

        ofi_m = (
            raw_ofi
            .to_frame(f"raw_ofi_{m:02d}")
            .set_index(df[COL_TIME])
            .resample(BUCKET, label="right", closed="right")
            .sum(min_count=1)
            .squeeze()
            .fillna(0.0)
        )

        resampled_ofi[f"ofi_level_{m:02d}"] = ofi_m

    # -------- average depth per interval (Eq. 3) ---------------------------
    depth_stats = (
        depth_per_update
        .to_frame("depth")
        .set_index(df[COL_TIME])
        .resample(BUCKET, label="right", closed="right")
        .agg(["sum", "count"])
    )
    avg_depth = (
        depth_stats[("depth", "sum")] /
        (depth_stats[("depth", "count")].replace(0, np.nan) * levels)
    ).rename("avg_depth").fillna(method="ffill").fillna(method="bfill")

    # -------- normalise OFI ------------------------------------------------
    ofi_df = pd.DataFrame({
        col: ser / avg_depth
        for col, ser in resampled_ofi.items()
    }).fillna(0.0)

    return ofi_df


# ── MAIN SCRIPT ────────────────────────────────────────────────────────────
def main(levels: int = 10) -> None:
    df_raw = load_lob(LOB_CSV, levels)
    ofi_matrix = compute_multi_level_ofi(df_raw, levels)

    ofi_matrix.to_csv(OUT_CSV, header=True)
    print(f"Multi‑Level OFI ({levels} levels) written to {OUT_CSV}")
    print(ofi_matrix.head())


if __name__ == "__main__":
    # example: compute 10‑level OFI with 1‑minute buckets
    main(levels=10)


Multi‑Level OFI (10 levels) written to multi_level_ofi.csv
                           ofi_level_00  ofi_level_01  ofi_level_02  \
ts_event                                                              
2024-10-21 11:55:00+00:00     -0.903537      2.780114      0.046335   
2024-10-21 11:56:00+00:00     -5.132746      3.595224      6.306603   
2024-10-21 11:57:00+00:00      0.908008      0.000000      5.475424   
2024-10-21 11:58:00+00:00      2.216406      2.624224      0.221641   
2024-10-21 11:59:00+00:00      2.078009      3.833643      2.491868   

                           ofi_level_03  ofi_level_04  ofi_level_05  \
ts_event                                                              
2024-10-21 11:55:00+00:00     -0.139006      0.115838      1.853409   
2024-10-21 11:56:00+00:00      0.124291      0.050637     -0.386682   
2024-10-21 11:57:00+00:00      0.000000      0.000000      0.000000   
2024-10-21 11:58:00+00:00      0.017731     -3.723561     -0.523072   
2024-10-21 11:59:

/var/folders/fh/3vnx049s0s1d3ns1h8j6xgb80000gn/T/ipykernel_80395/1624082097.py:122: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  avg_depth = (
